In [1]:
import json
from typing import Dict, Any

# =====================================================================
# 1. THE DATA INGESTION ENGINE
# =====================================================================
# Simulating a parsed corporate invoice structure (Extracted via layout tools)
MOCK_INVOICE_DATA = {
    "invoice_id": "INV-2026-8891",
    "vendor": "Prime Horizon Catering & Events",
    "total_amount": 12500.00,
    "line_items": [
        {"description": "Executive Luncheon Catering", "amount": 2500.00, "category": "Meals & Entertainment"},
        {"description": "Premium Open Bar (Corporate Gala)", "amount": 8000.00, "category": "Entertainment - Alcohol"},
        {"description": "Audio/Visual Equipment Rental", "amount": 2000.00, "category": "Operations"}
    ]
}

# =====================================================================
# 2. LOCAL TAX KNOWLEDGE BASE
# =====================================================================
# Internal tax compliance rules database (Localized, read-only)
TAX_COMPLIANCE_KNOWLEDGE_BASE = {
    "Meals & Entertainment": {
        "deductible_percentage": 0.50,
        "requires_attendee_list": True,
        "max_threshold_per_event": 5000.00
    },
    "Entertainment - Alcohol": {
        "deductible_percentage": 0.00,
        "requires_attendee_list": True,
        "max_threshold_per_event": 1000.00
    },
    "Operations": {
        "deductible_percentage": 1.00,
        "requires_attendee_list": False,
        "max_threshold_per_event": 50000.00
    }
}

# =====================================================================
# 3. THE MULTI-AGENT COMPLIANCE FRAMEWORK
# =====================================================================

def tax_rules_agent(category: str) -> Dict[str, Any]:
    """Agent 1: Specialized in fetching exact corporate tax rules for a category."""
    return TAX_COMPLIANCE_KNOWLEDGE_BASE.get(category, {
        "deductible_percentage": 0.00,
        "requires_attendee_list": True,
        "max_threshold_per_event": 0.00
    })

def compliance_auditor_agent(invoice: Dict[str, Any]) -> Dict[str, Any]:
    """Agent 2: Audits lines against tax code and flags compliance variance."""
    audit_report = {
        "invoice_id": invoice["invoice_id"],
        "vendor": invoice["vendor"],
        "total_claimed": invoice["total_amount"],
        "total_tax_deductible_allowance": 0.00,
        "flags": [],
        "audit_passed": True
    }
    
    for item in invoice["line_items"]:
        desc = item["description"]
        amount = item["amount"]
        category = item["category"]
        
        # Pull regulatory limits from Rules Agent
        rules = tax_rules_agent(category)
        
        # Calculate allowed deduction mapping
        allowed_deduction = amount * rules["deductible_percentage"]
        audit_report["total_tax_deductible_allowance"] += allowed_deduction
        
        # Check rule 1: Hard limits exceeded
        if amount > rules["max_threshold_per_event"]:
            audit_report["flags"].append({
                "item": desc,
                "category": category,
                "violation": "Threshold Exceeded",
                "details": f"Amount ${amount:.2f} exceeds limit of ${rules['max_threshold_per_event']:.2f}"
            })
            audit_report["audit_passed"] = False
            
        # Check rule 2: Complete non-deductibility warnings
        if rules["deductible_percentage"] == 0.00:
            audit_report["flags"].append({
                "item": desc,
                "category": category,
                "violation": "Non-Deductible Category",
                "details": "This expense item is marked as 0% deductible under current internal tax frameworks."
            })
            
    return audit_report

# =====================================================================
# 4. RUNNING THE PIPELINE
# =====================================================================
if __name__ == "__main__":
    print("--- Initiating Autonomous Tax Audit Simulation ---")
    
    # Process invoice through the agent loop
    final_report = compliance_auditor_agent(MOCK_INVOICE_DATA)
    
    # Output the structured audit findings
    print(json.dumps(final_report, indent=4))

--- Initiating Autonomous Tax Audit Simulation ---
{
    "invoice_id": "INV-2026-8891",
    "vendor": "Prime Horizon Catering & Events",
    "total_claimed": 12500.0,
    "total_tax_deductible_allowance": 3250.0,
    "flags": [
        {
            "item": "Premium Open Bar (Corporate Gala)",
            "category": "Entertainment - Alcohol",
            "violation": "Threshold Exceeded",
            "details": "Amount $8000.00 exceeds limit of $1000.00"
        },
        {
            "item": "Premium Open Bar (Corporate Gala)",
            "category": "Entertainment - Alcohol",
            "violation": "Non-Deductible Category",
            "details": "This expense item is marked as 0% deductible under current internal tax frameworks."
        }
    ],
    "audit_passed": false
}
